# Chronos-2 — PV-Prognose mit Wetter-Kovariate

Rolling 2-Tage-Forecasts über Sep–Dez 2023 (60 Fenster), GHI als bekannte
Zukunfts-Kovariate. Vergleich gegen lineare Regression aus dem vorigen Notebook.

Modell: [`amazon/chronos-2`](https://huggingface.co/amazon/chronos-2), CPU-Inferenz.
Daten: PV-Live (Fraunhofer ISE) + Open-Meteo Historical-Forecast, beide CC-BY-4.0.


In [1]:
import warnings; warnings.filterwarnings("ignore")
import time, numpy as np, pandas as pd, torch
import plotly.graph_objects as go
from pathlib import Path
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from chronos import BaseChronosPipeline

DATA = Path.home() / "1_data" / "temp"
pv  = pd.read_csv(DATA / "pv_data_year_stuttgart_0.6kwp.csv",
                  comment="#", parse_dates=["datetime_utc"], index_col="datetime_utc")
rad = pd.read_csv(DATA / "historical_weather_forecast_radiation_stuttgart.csv",
                  comment="#", parse_dates=["datetime_utc"], index_col="datetime_utc")
df = pd.concat([pv["pv_w"], rad["ghi_wm2"]], axis=1).dropna()

CTX_LEN, PRED_LEN = 3000, 192
EVAL_START = pd.Timestamp("2023-09-01 00:00", tz="UTC")
EVAL_END   = pd.Timestamp("2023-12-30 23:45", tz="UTC")


In [2]:
# Lineare Regression als Baseline — trainiert auf allen Punkten vor dem Eval-Fenster
buffer_start = EVAL_START - pd.Timedelta(minutes=15 * CTX_LEN)
train_df = df.loc[: buffer_start - pd.Timedelta("15min")]
lr = LinearRegression().fit(train_df[["ghi_wm2"]], train_df["pv_w"])
print(f"LR: slope={lr.coef_[0]:.3f},  intercept={lr.intercept_:.2f}")

# Chronos-2 rolling Forecasts mit GHI-Kovariate
pipe = BaseChronosPipeline.from_pretrained(
    "amazon/chronos-2", device_map="cpu", torch_dtype=torch.float32)

fc_starts = pd.date_range(
    EVAL_START, EVAL_END - pd.Timedelta(minutes=15 * (PRED_LEN - 1)),
    freq=f"{PRED_LEN * 15}min", tz="UTC")

idx_all, y_all, lr_all, c2_med, c2_lo, c2_hi = [], [], [], [], [], []
t0 = time.time()
for fs in fc_starts:
    fe = fs + pd.Timedelta(minutes=15 * (PRED_LEN - 1))
    ctx  = df.loc[fs - pd.Timedelta(minutes=15 * CTX_LEN) : fs - pd.Timedelta("15min")]
    test = df.loc[fs : fe]
    if len(ctx) != CTX_LEN or len(test) != PRED_LEN:
        continue
    inp = [{
        "target":            ctx["pv_w"].values.astype(np.float32),
        "past_covariates":   {"ghi_wm2": ctx["ghi_wm2"].values.astype(np.float32)},
        "future_covariates": {"ghi_wm2": test["ghi_wm2"].values.astype(np.float32)},
    }]
    q, _ = pipe.predict_quantiles(inp, prediction_length=PRED_LEN,
                                  quantile_levels=[0.1, 0.5, 0.9])
    qa = q[0][0].numpy()
    idx_all.extend(test.index)
    y_all.append(test["pv_w"].values)
    lr_all.append(np.clip(lr.predict(test[["ghi_wm2"]]), 0, None))
    c2_lo.append(np.clip(qa[:, 0], 0, None))
    c2_med.append(np.clip(qa[:, 1], 0, None))
    c2_hi.append(np.clip(qa[:, 2], 0, None))
print(f"{len(y_all)} Forecasts in {time.time()-t0:.1f}s")

y      = pd.Series(np.concatenate(y_all),    index=idx_all)
p_lr   = pd.Series(np.concatenate(lr_all),   index=idx_all)
p_c2   = pd.Series(np.concatenate(c2_med),   index=idx_all)
p_c2lo = pd.Series(np.concatenate(c2_lo),    index=idx_all)
p_c2hi = pd.Series(np.concatenate(c2_hi),    index=idx_all)


`torch_dtype` is deprecated! Use `dtype` instead!


LR: slope=0.550,  intercept=-1.25


`torch_dtype` is deprecated! Use `dtype` instead!


60 Forecasts in 8.0s


In [3]:
# Aggregierte Metriken über die gesamte Eval-Region
def met(p): return (mean_absolute_error(y, p), np.sqrt(mean_squared_error(y, p)))
mae_lr, rmse_lr = met(p_lr)
mae_c2, rmse_c2 = met(p_c2)
print(f"Eval: {len(y)} Punkte = {len(y)/96:.0f} Tage  ({EVAL_START.date()} – {EVAL_END.date()})\n")
print(f"{'Modell':<22s}  {'MAE [W]':>8s}  {'RMSE [W]':>9s}")
print(f"{'Lineare Regression':<22s}  {mae_lr:8.2f}  {rmse_lr:9.2f}")
print(f"{'Chronos-2 (+GHI)':<22s}  {mae_c2:8.2f}  {rmse_c2:9.2f}    "
      f"Δ {(mae_c2-mae_lr)/mae_lr*100:+.1f} % / {(rmse_c2-rmse_lr)/rmse_lr*100:+.1f} %")


Eval: 11520 Punkte = 120 Tage  (2023-09-01 – 2023-12-30)

Modell                   MAE [W]   RMSE [W]
Lineare Regression         18.67      42.46
Chronos-2 (+GHI)           14.82      38.17    Δ -20.7 % / -10.1 %


In [ ]:
# Beispiel-Fenster im FETS-Paper-Stil (Fig. 1):
#   - Blau solid    = historischer Kontext
#   - Rot dashed    = Chronos-2 Median-Forecast
#   - Rot shaded    = 10-90 %-Unsicherheitsband
#   - Grün solid    = Actual Values (Wahrheit)
#   - Orange dotted = Lineare Regression
#   - Vertikale Trennlinie an "Forecast Start"
WIN     = pd.Timestamp("2023-09-13 00:00", tz="UTC")
CTX_VIS = 3 * 96                                       # 3 Tage Kontext im Plot
sl      = slice(WIN, WIN + pd.Timedelta(minutes=15 * (PRED_LEN - 1)))
hist    = df["pv_w"].loc[WIN - pd.Timedelta(minutes=15 * CTX_VIS) : WIN - pd.Timedelta("15min")]

fig = go.Figure()

# Unsicherheitsband zuerst (sonst überdecken Linien es)
fig.add_scatter(x=p_c2hi.loc[sl].index, y=p_c2hi.loc[sl].values, mode="lines",
                line=dict(width=0), showlegend=False, hoverinfo="skip")
fig.add_scatter(x=p_c2lo.loc[sl].index, y=p_c2lo.loc[sl].values, mode="lines",
                line=dict(width=0), fill="tonexty",
                fillcolor="rgba(220,20,60,0.18)",
                name="Uncertainty (10–90 %)", hoverinfo="skip")

# Linien
fig.add_scatter(x=hist.index, y=hist.values, mode="lines",
                line=dict(color="#1f4ea1", width=1.6),
                name="Historical Data")
fig.add_scatter(x=y.loc[sl].index, y=y.loc[sl].values, mode="lines",
                line=dict(color="#2e8b57", width=1.8),
                name="Actual Values")
fig.add_scatter(x=p_c2.loc[sl].index, y=p_c2.loc[sl].values, mode="lines",
                line=dict(color="#dc143c", width=1.8, dash="dash"),
                name="Chronos-2 Forecast")
fig.add_scatter(x=p_lr.loc[sl].index, y=p_lr.loc[sl].values, mode="lines",
                line=dict(color="#e07b00", width=1.6, dash="dot"),
                name="Linear Regression")

# Trennlinie + Context/Horizon-Labels
fig.add_vline(x=WIN, line=dict(color="#888", width=1, dash="dash"))
fig.add_annotation(x=WIN, y=1.02, yref="paper", showarrow=False,
                   text="Forecast Start", font=dict(size=11, color="#666"),
                   xanchor="left", xshift=4)
fig.add_annotation(x=hist.index[len(hist) // 2], y=-0.18, yref="paper", showarrow=False,
                   text="Context", font=dict(size=11, color="#1f4ea1"))
fig.add_annotation(x=p_c2.loc[sl].index[PRED_LEN // 2], y=-0.18, yref="paper", showarrow=False,
                   text="Horizon", font=dict(size=11, color="#dc143c"))

# Paper-Stil: weißer Hintergrund, nur horizontale Grid-Linien, dezente Achsen
fig.update_layout(
    template="simple_white",
    height=420, width=900,
    margin=dict(t=60, b=70, l=70, r=20),
    legend=dict(orientation="h", yanchor="bottom", y=1.04,
                xanchor="left", x=0, font=dict(size=11),
                bgcolor="rgba(255,255,255,0)"),
    xaxis=dict(title="", showgrid=False, ticks="outside"),
    yaxis=dict(title="PV power [W]", showgrid=True, gridcolor="#eee",
               zeroline=False, ticks="outside"),
)
fig.show()
